In [1]:
from pathlib import Path
import numpy as np
import re
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

import math
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from tensorflow.keras.models import load_model



2026-05-05 19:03:43.061645: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-05 19:03:43.068063: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-05 19:03:43.322662: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-05 19:03:44.485856: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

In [2]:
# key=testname, val= (static_folder, dynamic_folder)

ROOT = Path("data/hf_dataset/pcap_csvs/split_csv")


test_dict = {
    "RoomS_LocA_ch011_20MHz": ("20_idle_ch11", "20_walking_ch11"),
    "RoomS_LocA_ch157_40MHz": ("40_idle", "40_walking"),
    "RoomS_LocA_ch157_80MHz": ("80_idle", "80_walking"),
    "RoomS_LocB_ch157_40MHz": ("40_idle_blocked", "40_walking_blocked"),
    "RoomS_LocB_ch157_80MHz": ("80_idle_blocked", "80_walking_blocked"),
    # matt
    "RoomM_Loc5_ch011_20MHz": ("20_5t_static_idle_roomM", "20_5ft_motion_walking_roomM"),
    "RoomM_Loc5_ch157_40MHz": ("40_5ft_static_idle_roomM", "40_5ft_motion_walking_roomM"),
    "RoomM_Loc5_ch157_80MHz": ("80_5ft_static_idle_roomM", "80_5ft_motion_walking_roomM"),
    
    "RoomM_Loc10_ch011_20MHz": ("20_10ft_static_idle_roomM", "20_10ft_motion_walking_roomM"),
    "RoomM_Loc10_ch157_40MHz": ("40_10ft_static_idle_roomM", "40_10ft_motion_walking_roomM"),
    "RoomM_Loc10_ch157_80MHz": ("80_10ft_static_idle_roomM", "80_10ft_motion_walking_roomM"),
}

SUBCARRIERS_20MHz = 64
SUBCARRIERS_40MHz = 128
SUBCARRIERS_80MHz = 256


L = 1024  


def gather_paths(static_path, dynamic_path):
    ''' Return paths and labels for static + dynamic files'''
    static_files  = sorted(static_path.glob("*.csv"))
    dynamic_files = sorted(dynamic_path.glob("*.csv"))

    paths = static_files + dynamic_files
    labels = np.array([0]*len(static_files) + [1]*len(dynamic_files), dtype=np.int32)
    return paths, labels

def load_mag_csv(path, num_subcarriers, L=L):
    # function to load in the magnitude data 
    
    df = pd.read_csv(path)
    # (T, 49)
    X = df.to_numpy(dtype=np.float32)          
    X = np.log1p(np.abs(X))                  
    T, S = X.shape

    # sanity check here to make sure we ahve correct subcarrier len
    assert S == num_subcarriers, f"Expected {num_subcarriers} cols, got {S}"

    if T >= L:
        start = np.random.randint(0, T - L + 1)
        X = X[start:start+L]
    else:
        X = np.pad(X, ((0, L - T), (0, 0)), mode="edge")
        
    # normalize data read in
    mean = np.mean(X, axis=0, keepdims=True)
    std = np.std(X, axis=0, keepdims=True)
    
    
    X_out = (X - mean) / (std + 1e-8)

    return X_out



In [3]:
# Separating RoomS and RoomM for ease of testing, speeds up and prevents long retries
model_20 = load_model("models/Base_ch011_20MHz_2.keras.keras")
model_40 = load_model("models/Base_ch157_40MHz_2.keras")
model_80 = load_model("models/Base_ch157_40MHz_2.keras" )


for key, paths in test_dict.items():   
    # to speed up / review just do roomS
    if "RoomM" in key:
        continue
    print("--")
    print(key)
    print("--")
    static_folder, dynamic_folder = paths
    print(f"static: {static_folder}")
    print(f"dynamic: {dynamic_folder}")
    
    if "20MHz" in key:
        new_model = model_20
        SUBCARRIERS = SUBCARRIERS_20MHz
    elif "40MHz" in key:
        new_model = model_40
        SUBCARRIERS = SUBCARRIERS_40MHz
    elif "80MHz" in key:
        new_model = model_80    
        SUBCARRIERS = SUBCARRIERS_80MHz
        
    #new_model = load_model(fname)

    #new_model.summary()
    
    static_files = ROOT / static_folder
    dynamic_files = ROOT / dynamic_folder
    

    new_paths, new_labels = gather_paths(static_files, dynamic_files)
    X_new = np.stack([
        load_mag_csv(p, SUBCARRIERS) for p in new_paths
    ])

    loss, accuracy, auc, pr_auc, precision, recall = new_model.evaluate(X_new, new_labels)


    print(f"loss={loss}, acc={accuracy}, auc={auc}, pr_auc={pr_auc}, prec={precision}, recall={recall}")
    print("--")

2026-05-05 19:03:47.350749: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


--
RoomS_LocA_ch011_20MHz
--
static: 20_idle_ch11
dynamic: 20_walking_ch11
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6970 - auc: 0.6114 - loss: 0.9925 - pr_auc: 0.4232 - precision: 0.0000e+00 - recall: 0.0000e+00         
loss=0.9925236701965332, acc=0.6969696879386902, auc=0.6114130616188049, pr_auc=0.423162043094635, prec=0.0, recall=0.0
--
--
RoomS_LocA_ch157_40MHz
--
static: 40_idle
dynamic: 40_walking
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7558 - auc: 0.9688 - loss: 0.5835 - pr_auc: 0.9699 - precision: 1.0000 - recall: 0.4750                 
loss=0.5834892988204956, acc=0.7558139562606812, auc=0.96875, pr_auc=0.9698500633239746, prec=1.0, recall=0.4749999940395355
--
--
RoomS_LocA_ch157_80MHz
--
static: 80_idle
dynamic: 80_walking
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6047 - auc: 0.7965 - loss: 0.5926 - pr_auc: 0.8235 - precision: 0.5500 - recall: 0.8250                 
loss=0.5926058888435364, acc=0.604651153087616, auc=0.7964674234390259, p

In [ ]:

# exact same as above, just roomM
for key, paths in test_dict.items():   
    # to speed up / review skip others
    if "RoomM" not in key:
        continue
    print("--")
    print(key)
    print("--")
    static_folder, dynamic_folder = paths
    print(f"static: {static_folder}")
    print(f"dynamic: {dynamic_folder}")
    
    if "20MHz" in key:
        new_model = model_20
        SUBCARRIERS = SUBCARRIERS_20MHz
    elif "40MHz" in key:
        new_model = model_40
        SUBCARRIERS = SUBCARRIERS_40MHz
    elif "80MHz" in key:
        new_model = model_80    
        SUBCARRIERS = SUBCARRIERS_80MHz
    #new_model.summary()
    
    static_files = ROOT / static_folder
    dynamic_files = ROOT / dynamic_folder
    
  
    new_paths, new_labels = gather_paths(static_files, dynamic_files)
    X_new = np.stack([
        load_mag_csv(p, SUBCARRIERS) for p in new_paths
    ])

    loss, accuracy, auc, pr_auc, precision, recall = new_model.evaluate(X_new, new_labels)


    print(f"loss={loss}, acc={accuracy}, auc={auc}, pr_auc={pr_auc}, prec={precision}, recall={recall}")
    print("--")

--
RoomM_Loc5_ch011_20MHz
--
static: 20_5t_static_idle_roomM
dynamic: 20_5ft_motion_walking_roomM
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5556 - auc: 0.7750 - loss: 1.0383 - pr_auc: 0.5791 - precision: 0.0000e+00 - recall: 0.0000e+00
loss=1.0382754802703857, acc=0.5555555820465088, auc=0.7749999761581421, pr_auc=0.5791095495223999, prec=0.0, recall=0.0
--
--
RoomM_Loc5_ch157_40MHz
--
static: 40_5ft_static_idle_roomM
dynamic: 40_5ft_motion_walking_roomM
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9987 - auc: 1.0000 - loss: 0.0089 - pr_auc: 1.0000 - precision: 1.0000 - recall: 0.9974
loss=0.008947094902396202, acc=0.998711347579956, auc=1.0, pr_auc=1.0, prec=1.0, recall=0.9974358677864075
--
--
RoomM_Loc5_ch157_80MHz
--
static: 80_5ft_static_idle_roomM
dynamic: 80_5ft_motion_walking_roomM
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.0062 - pr_auc: 1.0000 - precision: 1.0000 - recall: 1.0000
loss=0.006199008785188198, acc=1.0, auc=